# Extract and Train Sparse Embeddings (w150 Dataset)
This notebook performs the GPU-intensive extraction of DINOv3 and ReID embeddings for 4 sampled frames per shot (instead of a single keyframe) on the w150 dataset. It then trains the Graph Transformer to see if this sparse temporal representation improves the clustering/false-positive issue.

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive
import torch

# ==============================================================================
# Step 1: Mount Drive and set environment variable
# ==============================================================================
print("[STEP 1] Mounting Google Drive...")
drive.mount('/content/drive')
os.environ["DRIVE_ROOT"] = "/content/drive/MyDrive/CCTV-Multiview-Project"
DRIVE_ROOT = Path(os.environ["DRIVE_ROOT"])


In [ ]:
# ==============================================================================
# Step 2: Verify GPU
# ==============================================================================
print("\n[STEP 2] Verifying GPU...")
if not torch.cuda.is_available():
    print("[FAIL] GPU is not available! Please change the runtime type to T4/A100 GPU and restart.")
    sys.exit(1)
print(f"[PASS] GPU detected: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==============================================================================
# Step 3: Setup Environment
# ==============================================================================
repo_dir = "/content/cctv-multiview-summarization"
if not os.path.exists(repo_dir):
    print(f"[INFO] Cloning repository to {repo_dir}...")
    !git clone https://github.com/Gautam-Shah306/cctv-multiview-summarization.git {repo_dir}

os.chdir(repo_dir)
!git fetch origin
!git checkout feature/stage1-object-detection
!git pull origin feature/stage1-object-detection
!pip install -q -r requirements-colab.txt

print("\n[INFO] Installing PyTorch Geometric...")
!pip install -q torch-geometric


In [ ]:
# ==============================================================================
# Step 4: Extract Sparse-Sampled Embeddings
# ==============================================================================
print("\n[STEP 4] Extracting Sparse-Sampled Embeddings...")
!python -m src.extract_sparse_embeddings


In [ ]:
# ==============================================================================
# Step 5: Run 5-Fold CV Training (Sparse w150 dataset)
# ==============================================================================
print("\n[STEP 5] Running Graph Transformer 5-Fold Training on Sparse Data...")
!python -m src.train_graph_transformer_w150_sparse


In [ ]:
# ==============================================================================
# Step 6: Verify Saved Models and Backup Data
# ==============================================================================
print("\n[STEP 6] Persisting Models & Sparse Embeddings to Google Drive...")
import shutil

# Copy models
for fold in range(1, 6):
    local_model = Path(f"models/graph_transformer_w150_sparse_fold{fold}.pt")
    drive_model = DRIVE_ROOT / "models" / f"graph_transformer_w150_sparse_fold{fold}.pt"
    drive_model.parent.mkdir(parents=True, exist_ok=True)

    if local_model.exists():
        shutil.copy2(local_model, drive_model)
        size_mb = drive_model.stat().st_size / (1024 * 1024)
        print(f"[PASS] Fold {fold} Model successfully copied to Drive: {drive_model} ({size_mb:.2f} MB)")
    else:
        print(f"[FAIL] Fold {fold} Model file was not generated by the training script.")

# Copy newly extracted embeddings back to Drive just in case we need them
dino_sparse = Path("data_manifests/training_features_dino_w150_sparse.npz")
reid_sparse = Path("data_manifests/training_features_reid_w150_sparse.npz")

for f in [dino_sparse, reid_sparse]:
    if f.exists():
        drive_target = DRIVE_ROOT / "data_manifests" / f.name
        drive_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, drive_target)
        size_mb = drive_target.stat().st_size / (1024 * 1024)
        print(f"[PASS] Sparse embeddings successfully copied to Drive: {drive_target} ({size_mb:.2f} MB)")
